In [0]:
import data.utilities.common as Commonlib

import pyspark.sql.functions as F
import yaml

from beertech_utils.data.integrity.integrity_check import EmptyDataFrameCheck
from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from data.utilities.medallion_table_utility import MedallionTableUtility as MTU
from data.utilities.publishers import SnowflakeExternalTable
from pyspark.sql.types import TimestampType

In [0]:
CONFIG_BASE_PATH = "configuration/"

config_selection_path = f"{CONFIG_BASE_PATH}"
config_entries = Commonlib.list_files(config_selection_path, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "configuration file")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}/{config_file}"

config = Commonlib.get_object_config(config_file_path)

In [0]:
# Instantiate medallion object
medallion = Medallion(config)

In [0]:
SOURCE_TABLES = {
    # silver source tables
    "aufk": MTU.get_fully_qualified_table_name("oracle", "crd", "vw_aufk_base_table"),
    "hist": MTU.get_fully_qualified_table_name("external", "static", "intrl_ord_hist"),
}

In [0]:
# mapping from silver tables to gold intrl_ord table
aufk_mappings = {
    "intrl_ord_cd": F.col("aufnr"),
    "intrl_ord_typ": F.col("auart"),
    "intrl_ord_dsc": F.col("ktext"),
    "cmpy_cd": F.col("bukrs"),
    "bus_area_cd": F.col("gsber"),
}

In [0]:
try:
    # Base DF
    aufk_df = spark.read.table(SOURCE_TABLES["aufk"])[
        "aufnr", "auart", "ktext", "bukrs", "gsber", "mod_tsp"
    ].withColumns(aufk_mappings)
    aufk_df = aufk_df.select(*aufk_mappings.keys(), "mod_tsp")
except Exception as e:
    medallion.logger.error(f"Error processing intrl_ord source table reads/joins. Error message: {e}")
    raise

In [0]:
# if the table exists run the scd type 2 update
if spark.read.table(medallion.table_lookup["gold"]).count() > 0:
    try:  # Transform and select final columns
        medallion.df = (
            aufk_df.withColumn("__created_tsp", F.lit(medallion.start_datetime).cast(TimestampType()))
            .withColumn("edw_start_tsp", F.expr("mod_tsp"))
            .withColumn("edw_end_tsp", F.lit(None))
        )
        # write gold table
        medallion.write_to_delta(load_type="scd_type_2", layer="gold")
        medallion.logger.info("write to gold started for internal_order using SCD TYPE 2 with source data")
    except Exception as e:
        medallion.logger.error(f"Error writing internal_order table to gold layer. Error message: {e}")
        raise

# otherwise do historical backfill
else:
    hist_df = spark.read.table(SOURCE_TABLES["hist"])[
        "intrl_ord_cd",
        "intrl_ord_typ",
        "intrl_ord_dsc",
        "cmpy_cd",
        "bus_area_cd",
        "edw_start_tsp",
        "edw_end_tsp",
    ]
    try:
        medallion.df = hist_df.withColumn("__created_tsp", F.lit(medallion.start_datetime).cast(TimestampType()))
        # write gold table
        medallion.write_to_delta(load_type="overwrite", layer="gold")
        medallion.logger.info("write to gold started for internal_order using OVERWRITE with historical backfill")
    except Exception as e:
        medallion.logger.error(f"Error writing internal_order table to gold layer. Error message: {e}")
        raise

In [0]:
# publish to external stage - snowflake
try:
    if config.get("publish_to_sf", False):
        pub = SnowflakeExternalTable(medallion.catalog, medallion.schema, medallion.name)
        pub.publish_ext_table()

except Exception as e:
    medallion.logger.error(e)
    raise

In [0]:
# compare legacy and modern datasets
try:
    medallion.compare_legacy_and_modern(
        config["name"],
        config["comparison_check"]["legacy_query"],
        config["comparison_check"]["modern_query"],
        medallion.key_columns,
        config.get("comparison_check", {}).get("completeness_lower_bound"),
        config.get("comparison_check", {}).get("equivalency_lower_bound"),
        config.get("comparison_check", {}).get("col_exclusions"),
        config.get("comparison_check", {}).get("auto_resolve_col_diffs"),
        config.get("comparison_check", {}).get("treat_nulls"),
    )
except KeyError as e:
    medallion.logger.warning(
        "Skipping comparison of legacy and modern because the required config is missing or invalid."
    )